# 推論 & ベンチマーク ノートブック (inference)

保存済みモデル (マージ済み or ベース+LoRA) を読み込み、
1. プロンプト取得 (インライン or YAML)
2. 連続対話 (ストリーミング任意)
3. 各ターンの TTFT / 生成速度 / トークン数 / メモリ計測
4. 終了後サマリ集計
を行うためのノートブック。

このノートは学習パイプライン `pipeline.ipynb` で生成した成果物を利用する想定です。

In [ ]:
# === GPU/環境検出 & 推奨設定ガイド (Colab 推論) ===
import torch, platform
from datetime import datetime

def detect_gpu():
    if not torch.cuda.is_available():
        return {'available': False}
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    return {'available': True, 'name': name, 'total_gb': round(total,2)}

info = detect_gpu()
print('[ENV]', datetime.utcnow().isoformat(), platform.platform())
print('GPU Info:', info)

if info.get('available'):
    name = info['name']
    mem = info['total_gb']
    if 'L4' in name:
        print('\n推奨(L4 推論モード):')
        print('- PRECISION_MODE = 4bit または auto (4bit優先)')
        print('- STREAM_TTFT = True で待機体験向上')
        print('- GEN_KW.max_new_tokens を短めに調整 (<=512)')
        print('- 長時間対話で会話履歴が肥大する場合: TRUNCATE_PROMPT_TOKENS の導入を検討')
    elif 'A100' in name and mem >= 40:
        print('\n推奨(A100 推論高速化):')
        print('- PRECISION_MODE = bf16')
        print('- 長文応答で max_new_tokens 引き上げ (<=1024)')
        print('- tokens_per_s 目標値を記録し比較')
    else:
        print('\n汎用推論推奨:')
        print('- PRECISION_MODE = auto (fp16/bf16)')
        print('- メモリ不足時は 4bit に変更')
else:
    print('GPU 未利用: CPU 推論 (低速)')

## 依存関係
- transformers / peft / torch / (任意) bitsandbytes / PyYAML / pynvml
- 未導入の場合は `pip install pyyaml peft transformers pynvml` などを実行してください。
- Colab 利用時は最初にランタイムタイプ(GPU)を確認。

In [ ]:
# === 3. パラメータブロック (編集可能セクション / Colab L4 推論向け推奨初期値) ===
# L4 (24GB) を想定しメモリ効率とレスポンスを重視した設定。
# 大文字変数は config_hash 対象。

import os
from pathlib import Path

DATA_DIR = '/contents/llm-lab-save'               # Colab など一時環境での永続化先    

# モデル参照 (ローカルパス or HF Hub モデルID)
MODEL_BASE_PATH = f'{DATA_DIR}/artifacts/base'              # ローカルが無ければ MODEL_BASE_REF
MODEL_BASE_REF = 'Qwen/Qwen3-14B'         # L4 でも収まる 7B 系 (14B は4bit量子化必須)
LORA_ADAPTER_PATH = f'{DATA_DIR}/artifacts/lora'            # 任意 (base+lora 用)
MERGED_MODEL_PATH = f'{DATA_DIR}/artifacts/merged'
MERGED_MODEL_REF = None                           # 公開マージモデル利用時に指定
LOAD_MODE = 'merged'                              # 'merged' | 'base+lora'
PRECISION_MODE = '4bit'                           # L4 では 4bit デフォルトで VRAM 節約
TRUST_REMOTE_CODE = False

# プロンプト取得設定
PROMPT_SOURCE = 'inline'                          # 'inline' | 'yaml'
PROMPT_YAML_PATH = f'{DATA_DIR}/prompts/sample.yaml'
INLINE_SYSTEM_PROMPT = 'あなたは有能なアシスタントです。'
INLINE_USER_PROMPTS = [
    'このモデルの主な用途を3点で示してください。',
    '上記内容をビジネス向け提案メールの導入パラグラフに再構成してください。'
]

# 生成パラメータ (短時間応答重視)
GEN_KW = {
    'max_new_tokens': 320,        # L4 で長すぎない程度
    'temperature': 0.7,
    'top_p': 0.9,
    'top_k': 40,
    'repetition_penalty': 1.05,
    'do_sample': True,
}

# ベンチマーク / ログ出力関連
BENCH_LOG_DIR = f'{DATA_DIR}/inference_logs'
CSV_FILENAME = 'bench.csv'
SESSION_SUMMARY_FILENAME = 'session_summary.json'
CONVERSATION_FILENAME = 'conversation.json'
PARAMS_DUMP_FILENAME_TEMPLATE = 'params_{hash}.json'
APPEND_CONFIG_PARAMS_JSON = True

# ループ & 実行挙動
SEED = 42
STREAM_TTFT = True                                # L4 で体感向上 (prefill遅延を隠蔽)
MAX_TURNS = None
EXIT_COMMAND = '/exit'
ECHO_INPUT = True
SHOW_INTERMEDIATE_STREAM = True
TOKENIZER_CHAT_TEMPLATE_FALLBACK = True

# トークナイザ/モデル周り微調整
TRUNCATE_PROMPT_TOKENS = 4096                      # L4 で極端な履歴肥大を防止 (None にすれば解除)
APPLY_SYSTEM_PREFIX = True

# メモリ計測
USE_PYNVML = True
PEAK_RESET_EACH_TURN = False

# 出力整形
DISPLAY_METRICS_TABLE = True
ROUND_DIGITS = 2

# 内部ユーティリティ挙動
SAFE_SERIALIZE_ERRORS = True
RESERVED_OPTIONS = {}

# ディレクトリ準備
Path(BENCH_LOG_DIR).mkdir(parents=True, exist_ok=True)
print('L4 推論向け推奨パラメータ設定完了 (必要に応じて編集可)')

In [ ]:
# === 4. config_hash 生成 (プレースホルダ) ===
import json, hashlib, inspect
def collect_param_block(globals_dict):
    out = {}
    for k,v in globals_dict.items():
        if k.isupper():
            try:
                json.dumps(v)
                out[k]=v
            except TypeError:
                out[k]=str(v)
    return out
PARAM_BLOCK = collect_param_block(globals())
_norm = json.dumps(PARAM_BLOCK, sort_keys=True, ensure_ascii=False)
CONFIG_HASH = hashlib.sha256(_norm.encode('utf-8')).hexdigest()[:16]
print('config_hash:', CONFIG_HASH)

## 5. YAML スキーマ (説明)
```yaml
# system, prompts, conversations は任意。
system: あなたは有能なアシスタントです。
prompts:
  - 製品Aの特徴を3点で要約してください。
  - 上記を顧客向け提案書の冒頭要約として改善してください。
# conversations が存在する場合はそれが優先。
conversations:
  - { role: system, content: あなたは有能なアシスタントです。 }
  - { role: user, content: こんにちは }
  - { role: assistant, content: こんにちは！ご用件をどうぞ。 }
```
正規化優先順位: conversations > prompts > inline。

In [ ]:
# === 6. YAML ロード & 正規化 実装 ===
import json, yaml
from typing import List, Dict, Any

SYSTEM_PROMPT_FROM_YAML = None
USER_PROMPTS_FROM_YAML: List[str] = []
CONVERSATION_SEED: List[Dict[str, str]] = []

class YamlPromptError(Exception):
    pass

def _validate_role(r: str):
    if r not in {"system","user","assistant"}:
        raise YamlPromptError(f"不正な role: {r}")

def load_and_normalize_yaml(path: str):
    with open(path, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f) or {}
    if not isinstance(data, dict):
        raise YamlPromptError('YAML ルートは mapping が必要')
    system = data.get('system')
    prompts = data.get('prompts')
    conversations = data.get('conversations')

    if system is not None and not isinstance(system, str):
        raise YamlPromptError('system は文字列である必要があります')
    if prompts is not None:
        if not isinstance(prompts, list) or any(not isinstance(x,str) for x in prompts):
            raise YamlPromptError('prompts は文字列リスト')
    if conversations is not None:
        if not isinstance(conversations, list):
            raise YamlPromptError('conversations はリスト')
        for turn in conversations:
            if not isinstance(turn, dict):
                raise YamlPromptError('conversation 要素は dict')
            if 'role' not in turn or 'content' not in turn:
                raise YamlPromptError('conversation 要素は role, content 必須')
            _validate_role(turn['role'])
            if not isinstance(turn['content'], str):
                raise YamlPromptError('content は文字列')

    # 正規化
    seed_conv: List[Dict[str,str]] = []
    if conversations:
        seed_conv = list(conversations)
        # system が conversation 先頭に無い場合補完
        if system and not (len(seed_conv)>0 and seed_conv[0]['role']=='system'):
            seed_conv.insert(0, {'role':'system','content':system})
        sys_final = None
        usr_prompts = prompts or []
    else:
        # conversations 未指定 -> prompts + system から構成
        sys_final = system
        usr_prompts = prompts or []

    return {
        'system': system if conversations else sys_final,
        'user_prompts': usr_prompts,
        'seed_conversation': seed_conv,
    }

if PROMPT_SOURCE == 'yaml':
    if not os.path.exists(PROMPT_YAML_PATH):
        raise FileNotFoundError(f'YAML ファイルが存在しません: {PROMPT_YAML_PATH}')
    _res = load_and_normalize_yaml(PROMPT_YAML_PATH)
    SYSTEM_PROMPT_FROM_YAML = _res['system']
    USER_PROMPTS_FROM_YAML = _res['user_prompts']
    CONVERSATION_SEED = _res['seed_conversation']
    print(f"YAML 読込完了: system={'あり' if SYSTEM_PROMPT_FROM_YAML else 'なし'}, prompts={len(USER_PROMPTS_FROM_YAML)}, seed_turns={len(CONVERSATION_SEED)}")
else:
    print('PROMPT_SOURCE != yaml のため YAML ロードをスキップ')

In [ ]:
# === 7. モデルロード 実装 (HFモデルID対応) ===
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
try:
    from peft import PeftModel
except ImportError:
    PeftModel = None

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

def _exists_dir(p):
    return p is not None and isinstance(p, str) and os.path.isdir(p)

def _select_source(local_path: str|None, ref: str|None):
    # ローカルディレクトリが存在すれば最優先、無ければ ref (HF Hub) を返す
    if local_path and _exists_dir(local_path):
        return local_path
    return ref

def resolve_dtype():
    if PRECISION_MODE == 'fp16':
        return torch.float16
    if PRECISION_MODE == 'bf16':
        return torch.bfloat16
    if PRECISION_MODE == 'auto':
        if torch.cuda.is_available():
            if torch.cuda.is_bf16_supported():
                return torch.bfloat16
            return torch.float16
        return torch.float32
    if PRECISION_MODE == '4bit':
        return None
    return torch.float32

DTYPE = resolve_dtype()
load_kwargs = {
    'trust_remote_code': TRUST_REMOTE_CODE,
}
if DTYPE is not None:
    load_kwargs['torch_dtype'] = DTYPE

quant_4bit_args = None
if PRECISION_MODE == '4bit':
    try:
        import bitsandbytes as bnb  # noqa: F401
        quant_4bit_args = dict(load_in_4bit=True, device_map='auto')
    except Exception as e:
        print('[WARN] 4bit 指定ですが bitsandbytes 読込失敗 -> fp16/bf16 fallback:', e)
        if torch.cuda.is_available():
            load_kwargs['torch_dtype'] = torch.float16
        else:
            load_kwargs['torch_dtype'] = torch.float32

if LOAD_MODE == 'merged':
    src = _select_source(MERGED_MODEL_PATH, MERGED_MODEL_REF)
    if src is None:
        raise FileNotFoundError('MERGED_MODEL_PATH/MERGED_MODEL_REF のいずれも利用できません')
    print(f'Loading merged model from: {src}')
    model = AutoModelForCausalLM.from_pretrained(src, device_map='auto', **load_kwargs, **(quant_4bit_args or {}))
    tokenizer = AutoTokenizer.from_pretrained(src, use_fast=True, trust_remote_code=TRUST_REMOTE_CODE)
else:
    base_src = _select_source(MODEL_BASE_PATH, MODEL_BASE_REF)
    if base_src is None:
        raise FileNotFoundError('MODEL_BASE_PATH/MODEL_BASE_REF のいずれも利用できません')
    if PeftModel is None:
        raise ImportError('peft がインストールされていません。`pip install peft` を実行してください。')
    adapter_path = LORA_ADAPTER_PATH
    if not _exists_dir(adapter_path):
        raise FileNotFoundError(f'LORA_ADAPTER_PATH が存在しません: {adapter_path}')
    print(f'Loading base model from: {base_src}')
    base_model = AutoModelForCausalLM.from_pretrained(base_src, device_map='auto', **load_kwargs, **(quant_4bit_args or {}))
    tokenizer = AutoTokenizer.from_pretrained(base_src, use_fast=True, trust_remote_code=TRUST_REMOTE_CODE)
    print('Attaching LoRA adapter...')
    model = PeftModel.from_pretrained(base_model, adapter_path)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Model & tokenizer loaded. dtype:', DTYPE, 'vocab_size:', tokenizer.vocab_size)

In [ ]:
# === 8. 計測ユーティリティ 実装 ===
import time, csv, math, threading, gc
from datetime import datetime
from transformers import TextIteratorStreamer

try:
    import pynvml
    if USE_PYNVML:
        pynvml.nvmlInit()
        NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
    else:
        NVML_HANDLE = None
except Exception as e:
    print('[INFO] pynvml 初期化失敗/未使用:', e)
    NVML_HANDLE = None

conversation = []  # {role, content}

# プロンプト正規化 (system + 過去 + 今回 user)
def build_prompt(system_prompt: str|None, conv: list, new_user: str):
    # chat_template が存在する場合は後段で tokenizer.apply_chat_template を利用するので
    # ここでは簡易的なフォールバック組立のみ
    parts = []
    if APPLY_SYSTEM_PREFIX and system_prompt:
        parts.append(f"[SYSTEM]\n{system_prompt}\n")
    for turn in conv:
        if turn['role'] == 'user':
            parts.append(f"[USER]\n{turn['content']}\n")
        elif turn['role'] == 'assistant':
            parts.append(f"[ASSISTANT]\n{turn['content']}\n")
    parts.append(f"[USER]\n{new_user}\n[ASSISTANT]\n")
    return '\n'.join(parts)

# メモリ計測
def get_memory_gb():
    if torch.cuda.is_available():
        cur = torch.cuda.memory_allocated() / (1024**3)
        reserved = torch.cuda.memory_reserved() / (1024**3)
        peak = torch.cuda.max_memory_allocated() / (1024**3)
        if NVML_HANDLE:
            try:
                info = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
                nvml_used = info.used / (1024**3)
                return {
                    'alloc_gb': round(cur,3),
                    'reserved_gb': round(reserved,3),
                    'peak_gb': round(peak,3),
                    'nvml_used_gb': round(nvml_used,3),
                }
            except Exception:
                pass
        return {
            'alloc_gb': round(cur,3),
            'reserved_gb': round(reserved,3),
            'peak_gb': round(peak,3),
        }
    else:
        return {}

# 生成 + 計測
@torch.inference_mode()
def generate_with_metrics(user_input: str, system_prompt: str|None):
    if STREAM_TTFT:
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    else:
        streamer = None

    # chat_template があるなら活用
    use_template = hasattr(tokenizer, 'chat_template') and tokenizer.chat_template and TOKENIZER_CHAT_TEMPLATE_FALLBACK
    if use_template:
        chat_like = conversation + [{'role':'user','content':user_input}]
        if system_prompt:
            # system を先頭挿入 (重複チェック)
            if not (len(chat_like)>0 and chat_like[0].get('role')=='system'):
                chat_like.insert(0, {'role':'system','content':system_prompt})
        prompt_text = tokenizer.apply_chat_template(chat_like, tokenize=False, add_generation_prompt=True)
    else:
        prompt_text = build_prompt(system_prompt, conversation, user_input)

    tokenized = tokenizer(prompt_text, return_tensors='pt')
    input_ids = tokenized['input_ids'].to(model.device)
    prompt_tokens = input_ids.shape[1]

    gen_kwargs = dict(**GEN_KW)
    gen_kwargs['input_ids'] = input_ids
    gen_kwargs['pad_token_id'] = tokenizer.pad_token_id
    gen_kwargs['eos_token_id'] = tokenizer.eos_token_id
    if streamer:
        gen_kwargs['streamer'] = streamer

    start = time.time()
    first_token_time = None
    output_tokens = []

    def _generate():
        with torch.inference_mode():
            model.generate(**gen_kwargs)

    thread = threading.Thread(target=_generate)
    thread.start()

    generated_text = ''
    if streamer:
        for piece in streamer:
            if first_token_time is None:
                first_token_time = time.time()
            if SHOW_INTERMEDIATE_STREAM:
                print(piece, end='', flush=True)
            generated_text += piece
        print()
        thread.join()
    else:
        thread.join()
        out = model.generate(**gen_kwargs)
        gen_out = out[0][prompt_tokens:]
        generated_text = tokenizer.decode(gen_out, skip_special_tokens=True)
        first_token_time = start  # 非ストリーム時 精度は粗い

    end = time.time()
    ttft_ms = (first_token_time - start)*1000 if first_token_time else None
    total_ms = (end - start)*1000
    gen_ms = total_ms - (ttft_ms if ttft_ms else 0)

    # 新規トークン数
    new_token_ids = tokenizer(generated_text, return_tensors='pt')['input_ids'][0]
    new_tokens = len(new_token_ids)
    tokens_per_s = new_tokens / (gen_ms/1000) if gen_ms>0 else math.nan

    mem = get_memory_gb()

    metrics = {
        'ttft_ms': round(ttft_ms, ROUND_DIGITS) if ttft_ms else None,
        'gen_ms': round(gen_ms, ROUND_DIGITS),
        'total_ms': round(total_ms, ROUND_DIGITS),
        'prompt_tokens': prompt_tokens,
        'new_tokens': new_tokens,
        'tokens_per_s': round(tokens_per_s, ROUND_DIGITS) if not math.isnan(tokens_per_s) else None,
        **{f'mem_{k}':v for k,v in mem.items()}
    }
    if PEAK_RESET_EACH_TURN and torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    return generated_text, metrics

print('Measurement utilities ready.')

In [ ]:
# === 9. 連続対話ループ 実装 ===
import pandas as pd

bench_csv_path = os.path.join(BENCH_LOG_DIR, CSV_FILENAME)

# 初期シード会話構築
if PROMPT_SOURCE == 'yaml':
    base_system = SYSTEM_PROMPT_FROM_YAML or INLINE_SYSTEM_PROMPT
    seed_prompts = USER_PROMPTS_FROM_YAML if USER_PROMPTS_FROM_YAML else INLINE_USER_PROMPTS
    if CONVERSATION_SEED:
        conversation.extend(CONVERSATION_SEED)
else:
    base_system = INLINE_SYSTEM_PROMPT
    seed_prompts = INLINE_USER_PROMPTS

if APPLY_SYSTEM_PREFIX and base_system and not (conversation and conversation[0]['role']=='system'):
    conversation.insert(0, {'role':'system','content':base_system})

# CSV ヘッダ作成
if not os.path.exists(bench_csv_path):
    with open(bench_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['timestamp','turn_idx','user_chars','prompt_tokens','new_tokens','ttft_ms','gen_ms','total_ms','tokens_per_s','mem_alloc_gb','mem_reserved_gb','mem_peak_gb','mem_nvml_used_gb','config_hash'])

print('対話を開始します。終了するには', EXIT_COMMAND, 'を入力してください。')
turn_idx = 0
seed_iter = iter(seed_prompts) if seed_prompts else iter([])

while True:
    if MAX_TURNS is not None and turn_idx >= MAX_TURNS:
        print('MAX_TURNS 到達で終了')
        break
    try:
        # シードプロンプトが残っていれば自動使用
        try:
            user_input = next(seed_iter)
            print(f'[AUTO PROMPT] {user_input}')
        except StopIteration:
            user_input = input('User> ').strip()
        if not user_input:
            print('空行スキップ')
            continue
        if user_input == EXIT_COMMAND:
            print('終了コマンド検出')
            break
        if ECHO_INPUT:
            print('[INPUT]', user_input)

        generated, m = generate_with_metrics(user_input, base_system)
        conversation.append({'role':'user','content':user_input})
        conversation.append({'role':'assistant','content':generated})

        # 表示
        print('\n=== Assistant ===')
        print(generated)
        print('--- Metrics ---')
        for k,v in m.items():
            print(f'{k}: {v}')

        # CSV 追記
        with open(bench_csv_path, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                datetime.utcnow().isoformat(),
                turn_idx,
                len(user_input),
                m['prompt_tokens'],
                m['new_tokens'],
                m['ttft_ms'],
                m['gen_ms'],
                m['total_ms'],
                m['tokens_per_s'],
                m.get('mem_alloc_gb'),
                m.get('mem_reserved_gb'),
                m.get('mem_peak_gb'),
                m.get('mem_nvml_used_gb'),
                CONFIG_HASH,
            ])
        turn_idx += 1
    except KeyboardInterrupt:
        print('キーボード割り込みで終了')
        break

print('対話ループ終了')

In [ ]:
# === 10. サマリ集計 実装 ===
import statistics
summary_json_path = os.path.join(BENCH_LOG_DIR, SESSION_SUMMARY_FILENAME)
conversation_json_path = os.path.join(BENCH_LOG_DIR, CONVERSATION_FILENAME)
params_dump_path = os.path.join(BENCH_LOG_DIR, PARAMS_DUMP_FILENAME_TEMPLATE.format(hash=CONFIG_HASH))

rows = []
if os.path.exists(bench_csv_path):
    import csv
    with open(bench_csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for r in reader:
            rows.append(r)
else:
    print('bench.csv が存在しないため空サマリ')

numeric_fields = ['ttft_ms','gen_ms','total_ms','tokens_per_s','new_tokens','prompt_tokens']
metrics_summary = {}
if rows:
    for field in numeric_fields:
        vals = [float(r[field]) for r in rows if r.get(field) not in (None,'','None')]
        if not vals:
            continue
        metrics_summary[field] = {
            'count': len(vals),
            'avg': round(sum(vals)/len(vals),2),
            'median': round(statistics.median(vals),2),
            'p95': round(sorted(vals)[int(len(vals)*0.95)-1],2) if len(vals)>=20 else None,
            'max': round(max(vals),2),
            'min': round(min(vals),2),
        }
else:
    print('行が無いため metrics_summary は空')

summary_obj = {
    'config_hash': CONFIG_HASH,
    'turns': len(rows),
    'metrics_summary': metrics_summary,
    'timestamp_utc': datetime.utcnow().isoformat(),
}

with open(summary_json_path, 'w', encoding='utf-8') as f:
    json.dump(summary_obj, f, ensure_ascii=False, indent=2)
print('サマリJSON保存:', summary_json_path)

with open(conversation_json_path, 'w', encoding='utf-8') as f:
    json.dump(conversation, f, ensure_ascii=False, indent=2)
print('会話履歴保存:', conversation_json_path)

if APPEND_CONFIG_PARAMS_JSON:
    with open(params_dump_path, 'w', encoding='utf-8') as f:
        json.dump(PARAM_BLOCK, f, ensure_ascii=False, indent=2)
    print('パラメータブロック保存:', params_dump_path)

print('サマリ集計完了')

In [ ]:
# === 11. クリーンアップ 実装 ===
print('クリーンアップ開始')
try:
    del model
except NameError:
    pass
try:
    del tokenizer
except NameError:
    pass

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print('CUDA キャッシュ解放完了')
else:
    print('CPU モードのため簡易クリーンアップのみ')
print('クリーンアップ終了')

---
拡張候補: Rouge / BLEU / BERTScore 評価セル, top-k/p スイープ, プロンプト比較バッチ処理 など。